# Set 05 – Lineare Regression mit scikit-learn

Die lineare Regression sagt einen kontinuierlichen Zahlenwert voraus. Wir erzeugen einen kontrollierten Datensatz, lernen eine Gerade und untersuchen Steigung, Achsenabschnitt, Residuen und Regressionsmetriken.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)

## 1. Kontrollierte Daten erzeugen

Wir simulieren Wohnungen. Der Mietpreis hängt ungefähr linear von der Wohnfläche ab:

Miete = 8,5 mal Fläche + 320 + Zufallsrauschen

Die echte Beziehung kennen wir hier nur, weil wir die Daten selbst erzeugen.

In [ ]:
n = 120
flaeche = rng.uniform(25, 130, n)
miete = 8.5 * flaeche + 320 + rng.normal(0, 85, n)

daten = pd.DataFrame({
    "flaeche_qm": flaeche.round(1),
    "miete_euro": miete.round(2),
})
display(daten.head())
display(daten.describe().round(2))

## 2. Daten visualisieren

Jeder Punkt ist eine Wohnung. Eine steigende Punktwolke spricht für einen positiven linearen Zusammenhang.

In [ ]:
ax = daten.plot.scatter(
    x="flaeche_qm", y="miete_euro", figsize=(8, 5),
    alpha=0.75, color="#4C78A8"
)
ax.set_title("Kontrollierte Mietdaten")
ax.set_xlabel("Wohnfläche in m²")
ax.set_ylabel("Monatsmiete in Euro")
plt.show()

## 3. Merkmale und Ziel trennen

X muss bei scikit-learn eine zweidimensionale Tabelle sein. Deshalb verwenden wir doppelte eckige Klammern. y ist eine eindimensionale Zielserie.

In [ ]:
X = daten[["flaeche_qm"]]
y = daten["miete_euro"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print("Training:", X_train.shape, "Test:", X_test.shape)

## 4. Modell erzeugen und fitten

fit(X_train, y_train) lernt die Parameter aus den Trainingsdaten. Ein Regressionsmodell besitzt danach coef_ für die Steigung und intercept_ für den Achsenabschnitt.

Wichtig: LinearRegression ist ein Schätzer und hat predict(), aber kein transform(). transform() gehört zu Vorverarbeitungsschritten wie StandardScaler. In einer Pipeline werden beide Arten von Schritten verbunden.

In [ ]:
modell = LinearRegression()
modell.fit(X_train, y_train)

m = modell.coef_[0]
b = modell.intercept_
print(f"Gelernte Steigung m: {m:.2f} Euro pro m²")
print(f"Gelernter Achsenabschnitt b: {b:.2f} Euro")
print(f"Gelernte Gleichung: Miete = {m:.2f} mal Fläche + {b:.2f}")

## 5. Vorhersagen berechnen

predict(X_test) verwendet die gelernten, nun festen Parameter. Die Vorhersage lässt sich auch manuell mit m mal x plus b nachrechnen.

In [ ]:
y_pred = modell.predict(X_test)

vergleich = pd.DataFrame({
    "flaeche_qm": X_test["flaeche_qm"],
    "tatsaechlich": y_test,
    "vorhergesagt": y_pred,
})
vergleich["residuum"] = vergleich["tatsaechlich"] - vergleich["vorhergesagt"]
display(vergleich.head().round(2))

manuell = m * X_test.iloc[0, 0] + b
print("Manuell:", round(manuell, 2))
print("scikit-learn:", round(y_pred[0], 2))

## 6. Regressionsmetriken

- MAE ist der mittlere absolute Fehler und bleibt in Euro gut interpretierbar.
- RMSE bestraft große Fehler stärker.
- R² beschreibt, welcher Anteil der Streuung durch das Modell erklärt wird. 1 ist perfekt; 0 entspricht ungefähr einer konstanten Mittelwertvorhersage.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f} Euro")
print(f"RMSE: {rmse:.2f} Euro")
print(f"R²:   {r2:.3f}")

## 7. Gelernte Gerade visualisieren

Die blauen Punkte waren beim Fitten sichtbar. Die orangefarbenen Testpunkte waren für das Modell neu.

In [ ]:
x_linie = pd.DataFrame({
    "flaeche_qm": np.linspace(daten["flaeche_qm"].min(), daten["flaeche_qm"].max(), 200)
})
y_linie = modell.predict(x_linie)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_train["flaeche_qm"], y_train, alpha=0.55, label="Training")
ax.scatter(X_test["flaeche_qm"], y_test, alpha=0.9, label="Test")
ax.plot(x_linie["flaeche_qm"], y_linie, color="black", linewidth=2.5, label="gelernte Gerade")
ax.set_xlabel("Wohnfläche in m²")
ax.set_ylabel("Monatsmiete in Euro")
ax.set_title("Lineare Regression")
ax.legend()
plt.show()

## 8. Residuen prüfen

Ein Residuum ist tatsächlicher Wert minus Vorhersage. Bei einem passenden linearen Modell sollten die Residuen ohne klares Muster um die Nulllinie streuen.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(y_pred, y_test - y_pred, color="#E45756", alpha=0.8)
ax.axhline(0, color="black", linestyle="--")
ax.set_xlabel("vorhergesagte Miete")
ax.set_ylabel("Residuum")
ax.set_title("Residuen der Testdaten")
plt.show()

## 9. Einen neuen Wert vorhersagen

Auch eine einzelne Beobachtung wird als Tabelle mit denselben Spalten übergeben.

In [ ]:
neue_wohnung = pd.DataFrame({"flaeche_qm": [80]})
neue_miete = modell.predict(neue_wohnung)[0]
print(f"Geschätzte Miete für 80 m²: {neue_miete:.2f} Euro")

## Grenzen

Die Gerade kann nur lineare Zusammenhänge abbilden. Ausreißer beeinflussen den Least-Squares-Fit stark. Außerdem ist eine Vorhersage weit außerhalb des beobachteten Flächenbereichs eine riskante Extrapolation.